## Inputs

In [ ]:
input_filename = "./object-manifest.csv"
output_filename = "./anvil_gen3_DRS_alias_info.csv"

## Implementation

In [ ]:
from csv import DictReader, DictWriter
from collections import OrderedDict

In [ ]:
class MissingGsUriException (Exception):
    pass

class Gen3FileInfoTransformer:
    ROW_NUMBER_COLUMN_NAME = "original_row_number"
    TARGET_PATH: str = "target_path"
    GA4GH_DRS_URI: str = "ga4gh_drs_uri"
    ANVIL_GEN3_COMPACT_IDENTIFIER = "dg.ANV0"

    def __init__(self, input_filename: str, output_filename: str):
        self.input_filename = input_filename
        self.output_filename = output_filename

    @staticmethod
    def _get_gs_keypath(gs_uri: str):
        key_path_start_index = gs_uri.find("/", 5)
        key_path = gs_uri[(key_path_start_index + 1):]
        return key_path

    @staticmethod
    def _keypath_to_target_path(keypath: str) -> str:
        target_path = "/" + keypath.replace("/", "_")
        return target_path

    def _add_target_path(self, row: dict):
        urls = row['urls']
        if not urls or "gs://" not in urls:
            raise MissingGsUriException(str(row))
        gs_uri = sorted(urls.split(" "))[0].strip()
        assert gs_uri.startswith("gs://"), f"urls: {urls} gs_uri:{gs_uri}"
        key_path = self._get_gs_keypath(gs_uri)
        target_path = self._keypath_to_target_path(key_path)
        row[self.TARGET_PATH] = target_path

    def _add_ga4gh_drs_uri(self, row: dict):
        drs_id = row['guid']
        drs_uri = f"drs://{self.ANVIL_GEN3_COMPACT_IDENTIFIER}:{drs_id}"
        row[self.GA4GH_DRS_URI] = drs_uri

    def transform(self):
        with open(self.input_filename, newline='') as input_file:
            with open(self.output_filename, 'w', newline='') as output_file:
                reader = DictReader(input_file)
                writer = None
                row_counter = 0
                for row in reader:
                    row_counter += 1
                    try:
                        if not writer:
                            fieldnames = [self.ROW_NUMBER_COLUMN_NAME] + reader.fieldnames + [self.TARGET_PATH, self.GA4GH_DRS_URI]
                            writer = DictWriter(output_file, fieldnames)
                            writer.writeheader()
                        row[self.ROW_NUMBER_COLUMN_NAME] = f"{row_counter:06d}"
                        self._add_target_path(row)
                        self._add_ga4gh_drs_uri(row)
                        writer.writerow(row)
                    except MissingGsUriException as ex:
                        print(f"No 'gs://' URI found, excluding row: {row}")
                        print(MissingGsUriException)
                        continue;



## Main Execution


In [ ]:
transformer = Gen3FileInfoTransformer(input_filename, output_filename)

In [ ]:
transformer.transform()

In [ ]:
print("Done")